In [1]:
import re
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import nltk
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from tqdm import tqdm 

In [ ]:
# nltk.download('punkt')
# nltk.download('averaged_perceptron_tagger')

nltk.download('punkt'): Эта команда загружает модуль punkt, который используется для токенизации текста. Он позволяет разбивать текст на предложения и слова. Это особенно полезно, когда вы работаете с текстами на естественном языке, так как токенизация является важным первым шагом в большинстве задач NLP.    
nltk.download('averaged_perceptron_tagger'): Эта команда загружает предобученную модель для разметки частей речи (POS tagging). Модель averaged_perceptron_tagger позволяет определять, к какой части речи принадлежит каждое слово в тексте (например, существительное, глагол, прилагательное и т.д.). Это полезно для анализа структуры предложений и понимания контекста слов.   

- t_scores. t-оценка показывает, насколько значимо отклонение частоты совместного появления пары слов от ожидаемой частоты, если бы они были независимыми.
- Формула MI показывает, насколько часто пара слов встречается вместе по сравнению с тем, как часто они встречаются по отдельности.
- Странность (weirdness) оценивает, насколько необычным или редким является использование определенного слова или фразы в контексте определенного корпуса текстов.

In [25]:
def split_articles_by_title(file_path):
    with open(file_path, encoding="utf-8") as file:
        content = file.read()
    #разделяем содержимое по вхождениям "Title:" с сохранением этого слова в начале каждого блока
    articles = content.split("\nTitle:")
    #очищаем каждую статью от лишних пробелов и символов
    articles = [article.replace('\n', ' ').strip() for article in articles]
    return articles

def normalize_english(text):
    """Эта функция принимает текст, токенизирует его (разбивает на слова) и фильтрует токены по частям речи.
       Она оставляет только прилагательные и глаголы, а также существительные, приводя их к нижнему регистру.  """
    tokens = word_tokenize(text)
    words = []
    for t in tokens:
        pos = pos_tag([t])[0][1] #функция pos_tag спользуется для определения части речи токена t. Она возвращает список кортежей, где каждый кортеж содержит токен и его тег части речи. Мы берем первый элемент [0] и второй элемент [1], чтобы получить только тег части речи.
        if pos in ['JJ', 'JJR', 'JJS', 'NN', 'NNS', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ']:
            words.append(t.lower())
    return words

def calc_MI_tscore(cnt_articles_1, cnt_articles_2, cntv1, cntv2, thr):
    """Эта функция рассчитывает два показателя: MI (Mutual Information) и t-score для пар слов (двухсловных терминов).
       Она принимает матрицы частот для однословных и двусловных терминов, а также векторизаторы и пороговое значение thr.
       Для каждой пары слов из двусловных терминов, если их частота меньше порога, она пропускает эту пару.
       Затем она вычисляет MI и t-score и сохраняет их в словарях mis и t_scores.  """
    mis = {}
    t_scores = {}
    corp_len = cnt_articles_1.sum()
    for pair, index2 in tqdm(cntv2.vocabulary_.items()): #цикл проходит по всем парам слов из словаря двусловных терминов
        freq12 = cnt_articles_2[0, index2] #получаем частоту двусловной терминов
        if freq12 < thr:
            continue
        words = pair.split(' ')
        freq1 = cnt_articles_1[0, cntv1.vocabulary_[words[0]]]
        freq2 = cnt_articles_1[0, cntv1.vocabulary_[words[1]]]
        mis[pair] = freq12 / (freq1 * freq2)
        t_scores[pair] = (freq12 - (freq1 * freq2) / corp_len) / np.sqrt(freq12)
    return mis, t_scores


def calc_weirdness(cnt_articles_1, cnt_articles_2, cnt_articles_CS_1, cnt_articles_CS_2, cntv1, cntv2, cntv_CS_1, cntv_CS_2):
    """Рассчитывает weirdness для слов."""
    weir_1 = {}
    weir_2 = {}
    
    for word, index in tqdm(cntv1.vocabulary_.items()):
        if word in cntv_CS_1.vocabulary_.keys():
            w = cnt_articles_1[0, index] / (cnt_articles_CS_1[0, cntv_CS_1.vocabulary_[word]] + 1) 
        else:
            w = cnt_articles_1[0, index]
        weir_1[word] = w
           
    for word, index in tqdm(cntv_CS_2.vocabulary_.items()):
        if word in cntv2.vocabulary_.keys():
            w = cnt_articles_CS_2[0, index] / (cnt_articles_CS_2[0, cntv_CS_2.vocabulary_[word]] + 1)  
        else:
            w = cnt_articles_CS_2[0, index]
    weir_2[word] = w
    
    return sorted(weir_1.items(), key=lambda x: x[1], reverse=True)[:100], sorted(weir_2.items(), key=lambda x: x[1], reverse=True)[-100:]


file_path = "/Users/MAC/Desktop/Компьютерная лингвистика/articles.txt"
articles = split_articles_by_title(file_path)

file_path_CS = "/Users/MAC/Desktop/Компьютерная лингвистика/articles CS.txt"
articles_CS = split_articles_by_title(file_path_CS)

# Применяем нормализацию к каждой статье
normalized_articles = [' '.join(normalize_english(article)) for article in articles]

normalized_articles_CS = [' '.join(normalize_english(article)) for article in articles_CS]

# Векторизация
cntv12 = CountVectorizer(ngram_range=(1, 1)) #это векторизатор, который будет использоваться для создания векторов частот однословных терминов (униграмм).
cntv22 = CountVectorizer(ngram_range=(2, 2))

cnt_articles_1 = cntv12.fit_transform(normalized_articles) #fit: обучает векторизатор на данных, извлекая уникальные слова и создавая словарь.
#transform: преобразует текстовые данные в матрицу частот, где строки представляют документы, а столбцы — слова из словаря
cnt_articles_2 = cntv22.fit_transform(normalized_articles)

#для статей КШ
cntv_CS_1 = CountVectorizer(ngram_range=(1, 1))
cntv_CS_2 = CountVectorizer(ngram_range=(2, 2))

cnt_articles_CS_1 = cntv_CS_1.fit_transform(normalized_articles_CS)
cnt_articles_CS_2 = cntv_CS_2.fit_transform(normalized_articles_CS)

#расчет MI и t-score
mis, t_scores = calc_MI_tscore(cnt_articles_1, cnt_articles_2, cntv12, cntv22, 10)

#расчет weirdness
weirdness_1, weirdness_2 = calc_weirdness(cnt_articles_1, cnt_articles_2, cnt_articles_CS_1, cnt_articles_CS_2, cntv1, cntv2, cntv_CS_1, cntv_CS_2)

#сортировка и вывод
sorted_mis = sorted(mis.items(), key=lambda x: x[1], reverse=True)[:100]
sorted_t_score = sorted(t_scores.items(), key=lambda x: x[1], reverse=True)[:100]

display("Top 100 MI:")#Слова с высоким значением MI (например, "neural network" и "et al") указывают на сильную зависимость между ними
display(sorted_mis)
display("Top 100 t_score:")#Высокие значения t-оценки (например, "ml models" и "compchem ml") указывают на то, что эти фразы встречаются вместе значительно чаще,чем ожидалось
display(sorted_t_score)  
display("Top 100 Weirdness for first articles:")#Высокие значения странности (например, "deferred" и "interobserver") указывают на то, что эти слова используются значительно реже, чем ожидалось, что может указывать на их специфичность или редкость в данных статьях
display(weirdness_1)
display("Top 100 Weirdness for second articles:")#Значение 0.0 указывает на то, что фраза "courtesy elsevier" не является необычной и, вероятно, встречается очень часто
display(weirdness_2) #любезно предоставлено

100%|██████████████████████████████████| 31815/31815 [00:02<00:00, 12150.32it/s]


'Top 100 MI:'

[('neural network', 0.05238095238095238),
 ('et al', 0.05),
 ('so called', 0.030303030303030304),
 ('correlated wavefunction', 0.019548872180451128),
 ('been developed', 0.013333333333333334),
 ('machine learning', 0.012781954887218045),
 ('electronic structure', 0.011948529411764705),
 ('many body', 0.00819672131147541),
 ('have been', 0.007692307692307693),
 ('has been', 0.006818181818181818),
 ('molecules materials', 0.005720823798627002),
 ('data driven', 0.0038155802861685214),
 ('data sets', 0.0034077555816686253),
 ('data points', 0.002574002574002574),
 ('chemical space', 0.002496029044701611),
 ('wavefunction methods', 0.0024316109422492403),
 ('data set', 0.0020452885317750183),
 ('ml models', 0.0017973856209150326),
 ('training data', 0.0017069701280227596),
 ('chemical systems', 0.0016499175041247937),
 ('compchem methods', 0.001208300499080641),
 ('compchem ml', 0.0011909949164851125),
 ('be used', 0.0011054165410511506),
 ('ml algorithms', 0.001094391244870041),
 ('ml mod

'Top 100 t_score:'

[('ml models', 6.596078072420037),
 ('compchem ml', 6.348973156653509),
 ('have been', 5.908333337365674),
 ('be used', 5.69221994674808),
 ('data sets', 5.369248010423217),
 ('data set', 5.265444114451651),
 ('ml model', 5.139308865588811),
 ('compchem methods', 4.755854166951848),
 ('ml methods', 4.391157515457054),
 ('so called', 4.241230504684478),
 ('et al', 4.122275049260053),
 ('machine learning', 4.119856606336431),
 ('has been', 3.867261948209333),
 ('molecules materials', 3.8661644800478823),
 ('are used', 3.663888007536308),
 ('methods are', 3.639176242171079),
 ('correlated wavefunction', 3.603693577617985),
 ('electronic structure', 3.60251191417609),
 ('models be', 3.5307293519942284),
 ('data driven', 3.454957216818801),
 ('wavefunction methods', 3.449752614913896),
 ('training data', 3.443661195365976),
 ('ml algorithms', 3.4322197939716728),
 ('neural network', 3.315987043889627),
 ('chemical space', 3.3032412252379655),
 ('chemical systems', 3.2963778585110766),
 ('ml

'Top 100 Weirdness for first articles:'

[('deferred', 279.0),
 ('interobserver', 185.0),
 ('conflict', 135.0),
 ('unfractionated', 117.66666666666667),
 ('evolutive', 113.0),
 ('organon', 98.0),
 ('set', 91.0),
 ('copious', 58.0),
 ('cornerstone', 52.0),
 ('trainees', 47.0),
 ('situation', 45.0),
 ('lethal', 45.0),
 ('protection', 44.0),
 ('service', 44.0),
 ('loops', 42.0),
 ('redistributes', 42.0),
 ('nonrevascularized', 42.0),
 ('paradigms', 42.0),
 ('grants', 42.0),
 ('dynamics', 39.0),
 ('bmc', 38.0),
 ('organized', 38.0),
 ('future', 33.75),
 ('ease', 33.0),
 ('nonselective', 32.0),
 ('vasodilators', 31.0),
 ('bmi', 30.0),
 ('conceptualised', 30.0),
 ('nicholas', 30.0),
 ('verlag', 30.0),
 ('prospero', 29.0),
 ('rare', 29.0),
 ('tension', 28.0),
 ('webinars', 28.0),
 ('relied', 27.0),
 ('conclude', 27.0),
 ('next', 26.0),
 ('conclusion', 25.5),
 ('definitive', 25.0),
 ('levoheartshock', 25.0),
 ('worsens', 23.333333333333332),
 ('situations', 23.0),
 ('simple', 23.0),
 ('neutrophilic', 23.0),
 ('pharmacokinetics', 22.6

'Top 100 Weirdness for second articles:'

[('courtesy elsevier', 0.0)]